# 12 · 第三个工具：逐条核对，再重新排序

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → [12 第三个工具 ✅] → 13 起：标准接口、agent …
```

> **这一章的代码在合并请求 #14 里。** 这一章不调大模型，一分钱不花。

## 背景：分数分不出谁对

还是第 10 章那道 Perrault 题。按症状打分，"耳聋 67 型"排第 1，正确答案 Perrault 7 型排第 6。分数是这样算出来的（两个方向各取平均，再平均）：

| | 方向 A：病人的 5 个症状，它解释了多少 | 方向 B：它的症状，病人有多少 | 总分 |
|---|---|---|---|
| 耳聋 67 型 | 只有耳聋（5.38 ÷ 5 = 1.08） | 它只记了耳聋，全有（5.38） | **3.23** |
| Perrault 7 型 | 5 个全解释了（平均 4.01） | 11 个里病人有 5 个（平均 2.08） | **3.04** |

毛病出在方向 B：**一个病的资料写得越详细，越吃亏。** 耳聋 67 型只记了一个症状，反而占了便宜。

医生不会这样判断。医生会拿病人的症状**一条一条去核对**："这个病解释得了闭经吗？解释得了卵巢功能不全吗？"这一章做两件事：
1. 把"逐条核对"做成 agent 的第三个工具 `check_match`（计划里的 T4）
2. 用核对的结果给候选**重新排序**，看第 1 名能不能答对得更多

## 这一章做的 4 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 判断两个症状算不算对上 | HPO 的术语树 | 写法粗细不同也能对上 |
| 2 | 核对报告 | 解释了的 / 没解释的 / 矛盾的 / 缺的 | agent 拿到证据，不只是分数 |
| 3 | 先粗筛，再精排 | 前 30 名按"解释了几个症状"重排 | 第 1 名答对率 19% → 30.5% |
| 4 | 记成新的基线 | MLflow | agent 要过的线变高了 |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 两个症状算不算对上

### 在干什么
病人的一个症状，和疾病资料里的一个症状，判断算不算"对上了"。

### 背景
两边写的粗细常常不一样。病人记录写"双侧感音神经性耳聋"，疾病资料里只写"感音神经性耳聋"。人一看就知道对上了，因为前一个是后一个的一种。

让大模型读两串术语自己判断，常常不准：罕见病的术语有两万个，很多很冷门。

### 用了什么技术，为什么
用 HPO 的**术语树**判断（第 10 章第 1 步：每个术语都知道自己是谁"更具体的一种"）。只有三种情况算对上：

| 病人写的 | 资料里写的 | 关系 | 算对上吗 |
|---|---|---|---|
| 双侧感音神经性耳聋 | 双侧感音神经性耳聋 | 一样 | 算 |
| 双侧感音神经性耳聋 | 感音神经性耳聋 | 病人的更具体（是它的一种） | 算 |
| 听力障碍 | 感音神经性耳聋 | 病人的更笼统 | 算 |
| 感音神经性耳聋 | 癫痫 | 没有关系 | 不算 |

**有标准答案的事交给程序算，把结果交给模型；模型负责综合判断。** 好比让模型用计算器算数，而不是让它心算。

### 公司里什么情况怎么做
- **模型做不准、但有确定答案的事**（算数、查表、术语的上下级），都做成工具
- **规则要能说清楚**：这里只有三种算对上，每种一条测试

### 执行
用知识库判断上面表里的四对：

In [ ]:
from raredx.data import sources
from raredx.knowledge import hpo as hpo_knowledge

registry = sources.load_sources(ROOT / "evals/datasets.toml")
hpo = hpo_knowledge.load(sources.raw_dir(registry["hpo"], ROOT / "data"))

pairs = [("HP:0008619", "HP:0008619"), ("HP:0008619", "HP:0000407"),
         ("HP:0000365", "HP:0000407"), ("HP:0000407", "HP:0001250")]
for finding, feature in pairs:
    print(f"{hpo.term(finding)[1]:<44} vs {hpo.term(feature)[1]:<34} → {hpo.relation(finding, feature)}")

### 结果怎么看
- `same`：一样
- `more specific`：病人的更具体
- `less specific`：病人的更笼统
- `None`：没有关系，不算对上

### 面试怎么说
> "Whether a patient's finding matches a disease feature is decided by the ontology — the same term, or one a more specific kind of the other — not by the model. Exact questions go to deterministic tools; the model does the judgement."

---

## 第 2 步 · 核对报告：四类结果

### 在干什么
给 `check_match` 一个病人和一个病，它逐条核对，交回一份报告。

### 背景
第 10 章的工具只说"像不像"（一个分数）。agent 要判断、要向医生解释，需要知道**哪里像、哪里不像**。

### 用了什么技术，为什么
报告分四类：

| 类别 | 意思 | 例子（下面会看到） |
|---|---|---|
| 解释了的 | 病人的症状，这个病有 | Perrault 7 型：5 个全有 |
| 没解释的 | 病人的症状，这个病没有 | 耳聋 67 型：闭经、卵巢功能不全…… |
| 矛盾的 | 病人有，但这个病的文献查过、没有一个病人有 | 组氨酸血症："智力障碍 0/104" |
| 缺的 | 这个病一半以上病人有，这位病人却没记录 | Perrault 7 型：条索状卵巢 1/1 |

每一条都带出处（哪篇论文），agent 的结论可以引用。考试时同样遮住题目自己的论文（第 10 章那套，自动生效）。

### 公司里什么情况怎么做
- **给模型的不只是结论，还有证据**：模型才能写出"为什么"，医生才能核实
- **"缺的"只算一半以上病人有的症状**：偶尔出现的症状，病人没有很正常，不算疑点

### 执行
先看 Perrault 那道题的两个候选（这里用不遮的知识演示，就像真实门诊）：

In [ ]:
import mlflow
from raredx.config import settings
from raredx.data import cases as case_store
from raredx.tools.check_match import CheckMatch

mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment("playground")
check_match = CheckMatch(hpo)

by_id = {c.id: c for c in case_store.read(sources.processed_dir(registry["phenopacket-store"], ROOT / "data"))}

def show(report):
    print(f"{report.disease_name}：解释了 {len(report.explained)} / {report.n_findings} 个症状")
    for e in report.explained:
        print(f"   ✓ {e.finding.label}（这个病：{e.feature}，{e.frequency or '频率不知道'}）")
    for f in report.unexplained:
        print(f"   ✗ {f.label}（这个病没有）")
    for c in report.contradicted:
        print(f"   ! {c.finding.label}（这个病的文献：{c.frequency}，{', '.join(c.sources)}）")
    if report.missing:
        print(f"   缺的（这个病一半以上病人有）：" + "、".join(f"{m.feature.label} {m.frequency}" for m in report.missing))
    print()

case = by_id["PMID_39701103_Proband_F3"]
show(check_match(case.hpo_ids, "OMIM:610265"))   # 耳聋 67 型，第 10 章排第 1
show(check_match(case.hpo_ids, case.disease_id))  # Perrault 7 型，正确答案

再看一个"矛盾"的例子。另一道题，病人 7 个症状里有"智力障碍"；按症状打分，"组氨酸血症"排在前 3 名里：

In [ ]:
other = by_id["PMID_11555793_sister_BV"]
show(check_match(other.hpo_ids, "OMIM:235800"))  # 组氨酸血症
print("这道题的正确答案：", other.disease_label)

### 结果怎么看
- **耳聋 67 型**：只解释了耳聋（1/5），闭经、卵巢功能不全、智力障碍、发育迟缓都是 ✗
- **Perrault 7 型**：5 个全是 ✓。"缺的"有条索状卵巢、两个激素升高等：资料里有，这位病人的记录里没有。这不一定是疑点，可能只是没查
- **组氨酸血症**：7 个症状一个都没解释（0/7）；有一条矛盾：它的文献查过 104 个病人，**没有一个有智力障碍**，而这位病人有；还缺了它的标志性表现：血里组氨酸升高，50 个病人里 50 个有，这位病人没有。证据都在说"不是它"，正确答案确实是另一个病（脑肌酸缺乏综合征 3 型）

### 面试怎么说
> "The match check returns evidence, not a score: findings explained — and how, exactly or through the ontology — findings unexplained, findings the disease's sources looked for and never saw, and the disease's frequent features the patient lacks, each with its source."

---

## 第 3 步 · 先粗筛，再精排

### 在干什么
用核对结果给候选重新排序：第 10 章的工具先给出**前 30 名**，再按"**解释了病人几个症状**"重排；解释得一样多的，保持原来的顺序。

### 背景
背景那张表说过：第 10 章的分数，让"资料很短、全对上"的病占便宜。"解释了病人几个症状"不会：耳聋 67 型解释 1 个，Perrault 7 型解释 5 个，一比就分出来了。

### 用了什么技术，为什么
**两阶段：先粗筛，再精排。** 搜索引擎、推荐系统、RAG（先检索、再重排）都是这个做法：
- **粗筛**：快、宽。8,000 多个病 → 30 个（第 10 章的工具，每道题 0.03 秒）
- **精排**：细、准。30 个逐条核对 → 前 10 名

为什么不直接把 8,000 多个病都逐条核对？没必要：正确答案多半已经在前 30 名里，粗筛的工作就是不漏掉它。

在 dev-200 上试过几种做法（都遮住论文）：粗筛取 10、30、50 个；直接数"解释了几个"，或者按症状的少见程度加权；把"矛盾"算进去或不算。差别都在误差范围内，取 30 个略好一点。**差不多的时候，选最简单的**：取 30 个，直接数个数，不算矛盾。

### 公司里什么情况怎么做
- **两阶段是标准做法**：第一阶段只管"别漏"，第二阶段只管"排准"
- **复杂的规则要用分数换**：加权、加罚分，如果分数没有明显变好，就不加。规则越简单，越容易解释、越不容易在 dev-200 上"调过头"

### 执行
先看 Perrault 那道题重排后的样子（不遮的知识，演示用）：

In [ ]:
from raredx.baselines.phenotype_rank import PhenotypeRank
from raredx.baselines.rank_then_check import RankThenCheck
from raredx.evals.systems import question_of
from raredx.tools.rank_diseases import RankDiseases

rank_diseases = RankDiseases(hpo)
before = PhenotypeRank(rank_diseases).predict(question_of(case))
after = RankThenCheck(rank_diseases, check_match).predict(question_of(case))
print("只按分数：    ", "；".join(c.disease_name for c in before[:3]))
print("先排名再核对：", "；".join(c.disease_name for c in after[:3]))

再正式考 dev-200（默认遮住论文，十几秒）：

In [ ]:
run("uv run raredx eval run --system rank-then-check --split dev-200")

### 结果怎么看
- Perrault 那道题：重排后 **Perrault 7 型排第 1**，Perrault 6 型第 2
- dev-200：

| | 第 1 名答对 | 前 3 名 | 前 10 名 |
|---|---|---|---|
| 只按分数（第 10 章） | 19.0% | 33.0% | 49.0% |
| **先排名再核对** | **30.5%** | **43.5%** | **54.5%** |

第 1 名答对率多了 11.5 个百分点，而且没用大模型、不花钱、每道题 0.04 秒。前 10 名也多了 5.5 个点：原来排在 11–30 名的正确答案，有一些被核对拉进了前 10。

### 面试怎么说
> "A two-stage design — similarity proposes 30 candidates, the match check reorders them by findings explained — lifts top-1 from 19% to 30.5% on the dev set with no model. Of the variants tried, the simplest did as well as any, so it is the one kept."

---

## 第 4 步 · 记成新的基线：agent 要过的线变高了

### 在干什么
把目前所有的系统放在一起比。

### 背景
第 10 章说过，agent 要过的线是"不用大模型能做到多好"。现在不用大模型能做得更好了，线也跟着变高。

### 公司里什么情况怎么做
- **永远和最强的简单方法比**：agent 要调大模型、要花钱、要几秒钟。它必须比这个免费、0.04 秒的方法好，才值得用
- **简单方法也要认真做**：基线做得太弱，后面的进步就是假的

### 执行

In [ ]:
run("uv run raredx eval runs --split dev-200")

### 结果怎么看
整理一下（都遮住论文）：

| 系统 | 第 1 名答对 | 前 10 名 | 用大模型吗 |
|---|---|---|---|
| 瞎猜 | 0.5% | 9.0% | 不用 |
| 大模型直接猜（第 09 章） | 14.5% | 28.0% | 用 |
| 只按分数（第 10 章） | 19.0% | 49.0% | 不用 |
| **先排名再核对（这一章）** | **30.5%** | **54.5%** | 不用 |

还记得第 11 章：有 41 道题的正确答案，遮住后在 HPO 里一个症状都没有。只靠 HPO 的方法，天花板大约 80%。

接下来 agent 要证明两件事：比"先排名再核对"答得更准；剩下那些新病，能靠查文献找回来一些。

### 面试怎么说
> "The bar for the agent is the strongest baseline without a model — rank-then-check at 30.5% top-1 and 54.5% top-10 — not the weakest; a model that costs money and seconds per case has to beat a free one that takes 40 milliseconds."

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 算不算对上 | HPO 术语树：一样 / 更具体 / 更笼统 | 有标准答案的事交给程序 |
| 2 核对报告 | 解释了的 / 没解释的 / 矛盾的 / 缺的，每条带出处 | 给模型证据，不只是分数 |
| 3 先粗筛再精排 | 前 30 名按"解释了几个症状"重排 | 第 1 名 19% → 30.5%，不用大模型 |
| 4 新基线 | MLflow 里一起比 | agent 要过的线变高 |

**一句话**：第三个工具把病人的症状和一个病逐条核对；用它给候选重新排序，第 1 名答对率从 19% 提高到 30.5%，不花一分钱。这成了 agent 要打败的新基线。

## 面试题

1. **Why not let the LLM compare the patient's findings with a disease itself?**
   > "Whether one HPO term is a kind of another is an exact question with an answer in the ontology; models get it wrong on rare terms. The tool computes it and hands the model evidence; the model weighs it."

2. **What is a two-stage ranker, and why use one here?**
   > "A cheap first stage proposes candidates for recall, an expensive second stage reorders them for precision — the shape of search, recommendation and retrieval-augmented generation. Similarity proposes 30 diseases; the match check reorders them."

3. **You tried several variants on the dev set. How did you choose?**
   > "All were within the confidence interval of each other, so I kept the simplest — a fixed pool of 30 and a plain count of findings explained. Simpler rules are easier to explain and less likely to be overfit to the dev set."

4. **What does your agent have to beat now?**
   > "Rank-then-check: 30.5% top-1, 54.5% top-10 on the dev set, free and 40 milliseconds per case. Beating the weakest baseline would prove nothing."

**下一章 13 · 把三个工具做成标准接口（MCP）**：MCP 是现在 agent 调用工具的通用标准。做完以后，各家的 agent（Claude、LangGraph 等）都能直接调用我们的工具。